# WAXAL ASR Challenge — First Submission (Zero-shot MMS baseline)

**Run this notebook on Google Colab with a GPU runtime**
(`Runtime > Change runtime type > T4 GPU`).

What this notebook does:
1. Installs deps and authenticates to Zindi (your password is entered via a
   masked `getpass` prompt **inside this Colab session** — it is never sent
   anywhere else, and Claude never sees it).
2. Downloads `Train.csv`, `Test.csv`, `SampleSubmission.csv` (and the
   official starter notebook, for reference) from the challenge.
3. Loads the matching audio for Lingala (`lin`), Shona (`sna`), and
   Luganda (`lug`) from `google/WaxalNLP` on Hugging Face and joins it to
   the challenge CSVs by `id`.
4. Runs **zero-shot inference** with Meta's `facebook/mms-1b-all` — it ships
   a dedicated CTC adapter for each of `lin`/`sna`/`lug`, so no fine-tuning
   is needed to get a real first score on the board.
5. Sanity-checks WER/CER on a held-out slice of `Train.csv`.
6. Builds `submission.csv` in the exact shape of `SampleSubmission.csv`.
7. (Optional, off by default) submits the file to Zindi.

This is a **baseline**, not the final model — MMS zero-shot gets you a real
leaderboard number fast. Fine-tuning Whisper or MMS on the WAXAL train
split (like the multilingual fine-tuning approach used in the
`afrivoices-asr-hack` project) is the natural next step to climb the board.

In [ ]:
# ============================================================
# 0. Install packages
# ============================================================
get_ipython().system('apt-get -qq install -y ffmpeg > /dev/null')
get_ipython().system('pip install -q -U "transformers>=4.46" "datasets[audio]>=2.20" accelerate jiwer soundfile librosa zindi pandas huggingface_hub')

In [ ]:
# ============================================================
# 1. Imports and config
# ============================================================
import os
import re
import unicodedata
import random
from getpass import getpass

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from jiwer import wer, cer
from transformers import AutoProcessor, Wav2Vec2ForCTC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "./dataset"
os.makedirs(DATA_DIR, exist_ok=True)

CHALLENGE_ID = "google-waxal-asr-challenge"  # Zindi slug from the URL
LANGS = ["lin", "lug", "sna"]  # Lingala, Luganda, Shona
MMS_MODEL_ID = "facebook/mms-1b-all"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU.")

## 2. Zindi auth + download

You'll be prompted for your Zindi **username** and **password** below.
The password box is masked (`getpass`) and lives entirely in this Colab
runtime's memory for this session only.

In [ ]:
from zindi.user import Zindian

zindi_username = input("Zindi username: ").strip()
zindi_password = getpass("Zindi password: ")

user = Zindian(username=zindi_username, fixed_password=zindi_password)
del zindi_password  # don't keep it around longer than needed

user.select_a_challenge(challenge_id=CHALLENGE_ID)
downloaded = user.download_dataset(destination=DATA_DIR)
print("Downloaded:", downloaded)

In [ ]:
# ============================================================
# 3. Inspect the challenge CSVs
# ============================================================
train_df = pd.read_csv(f"{DATA_DIR}/Train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/Test.csv")
sample_sub_df = pd.read_csv(f"{DATA_DIR}/SampleSubmission.csv")

print("Train.csv", train_df.shape)
display(train_df.head())
print("\nTest.csv", test_df.shape)
display(test_df.head())
print("\nSampleSubmission.csv", sample_sub_df.shape)
display(sample_sub_df.head())

print("\nColumns -> Train:", list(train_df.columns))
print("Columns -> Test:", list(test_df.columns))
print("Columns -> SampleSubmission:", list(sample_sub_df.columns))

# Try to find the language column automatically; fall back to inspecting ids.
lang_col = next((c for c in train_df.columns if "lang" in c.lower()), None)
if lang_col:
    print("\nLanguage distribution (train):")
    print(train_df[lang_col].value_counts())
else:
    print("\nNo obvious language column — inspect train_df.columns and adjust below.")

**Checkpoint:** confirm the printed columns above match what the code below
assumes before continuing. If `Train.csv`/`Test.csv` use different column
names than `id`/`transcription`/`language`, adjust `ID_COL`, `TEXT_COL`,
`LANG_COL` in the next cell.

In [ ]:
ID_COL = next((c for c in train_df.columns if c.lower() in ("id", "audio_id")), train_df.columns[0])
LANG_COL = lang_col or "language"
TEXT_COL = next((c for c in train_df.columns if c.lower() in ("transcription", "text", "target", "sentence")), None)

SUB_ID_COL = sample_sub_df.columns[0]
SUB_TEXT_COL = sample_sub_df.columns[1]

print(f"ID_COL={ID_COL!r}  LANG_COL={LANG_COL!r}  TEXT_COL={TEXT_COL!r}")
print(f"Submission columns: {SUB_ID_COL!r}, {SUB_TEXT_COL!r}")

## 4. Load WaxalNLP audio and join by id

`google/WaxalNLP` exposes one HF `datasets` config per language, e.g.
`lin_asr`, `lug_asr`, `sna_asr`, each with `train`/`validation`/`test`/
`unlabeled` splits and fields `id`, `speaker_id`, `audio`, `transcription`,
`language`, `gender`. We pull all rows for the three challenge languages
and index them by `id` so we can attach audio to the challenge CSVs.

In [ ]:
def load_audio_index(lang: str) -> dict:
    """id -> audio dict ({'array':..., 'sampling_rate':...}) for one language,
    across all splits (some challenge ids may fall in train/val/test)."""
    index = {}
    ds_dict = load_dataset("google/WaxalNLP", f"{lang}_asr")
    for split_name, split in ds_dict.items():
        if split_name == "unlabeled":
            continue
        for row in split:
            index[row["id"]] = row["audio"]
    print(f"{lang}: indexed {len(index)} ids across {list(ds_dict.keys())}")
    return index

audio_index = {}
for lang in LANGS:
    audio_index.update(load_audio_index(lang))

print(f"\nTotal indexed ids across all 3 languages: {len(audio_index)}")

# Sanity check overlap with the challenge ids.
sample_train_ids = set(train_df[ID_COL].astype(str).head(20))
sample_hf_ids = set(list(audio_index.keys())[:20])
print("\nSample Train.csv ids:", list(sample_train_ids)[:5])
print("Sample WaxalNLP ids: ", list(sample_hf_ids)[:5])
matched = train_df[ID_COL].astype(str).isin(audio_index.keys()).sum()
print(f"\nTrain.csv ids matched in WaxalNLP audio index: {matched}/{len(train_df)}")
if matched == 0:
    print("!! No ids matched — inspect the id formats above and adjust the "
          "matching logic (e.g. strip a language prefix/suffix) before continuing.")

In [ ]:
test_matched = test_df[ID_COL].astype(str).isin(audio_index.keys()).sum()
print(f"Test.csv ids matched in WaxalNLP audio index: {test_matched}/{len(test_df)}")

## 5. Zero-shot MMS-1b-all baseline

`facebook/mms-1b-all` ships a dedicated CTC adapter per language
(`adapter.lin.*`, `adapter.lug.*`, `adapter.sna.*` all confirmed present in
the model repo). We swap the adapter per language and decode greedily —
no fine-tuning needed for a first real leaderboard score.

In [ ]:
processor = AutoProcessor.from_pretrained(MMS_MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MMS_MODEL_ID).to(DEVICE)
model.eval()

_current_adapter = None

def set_language(lang: str):
    global _current_adapter
    if _current_adapter != lang:
        processor.tokenizer.set_target_lang(lang)
        model.load_adapter(lang)
        _current_adapter = lang


@torch.no_grad()
def transcribe(audio_arrays: list, lang: str, batch_size: int = 8) -> list:
    set_language(lang)
    outputs = []
    for i in range(0, len(audio_arrays), batch_size):
        batch = audio_arrays[i:i + batch_size]
        inputs = processor(batch, sampling_rate=16_000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(DEVICE)
        attn = inputs.get("attention_mask")
        if attn is not None:
            attn = attn.to(DEVICE)
        logits = model(input_values, attention_mask=attn).logits
        ids = torch.argmax(logits, dim=-1)
        outputs.extend(processor.batch_decode(ids))
    return outputs


def get_array(audio_field) -> np.ndarray:
    arr = np.asarray(audio_field["array"], dtype=np.float32)
    sr = audio_field.get("sampling_rate") or audio_field.get("sample_rate") or 16_000
    if sr != 16_000:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16_000)
    return arr

## 6. Sanity check on a held-out slice of Train.csv

Cheap gut-check before burning a submission: run the baseline on ~30
examples per language from `Train.csv` (which has ground truth) and report
WER/CER/weighted score with the same normalisation the leaderboard uses.

In [ ]:
def normalise(text: str) -> str:
    text = "" if text is None else str(text)
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def score(refs: list, hyps: list) -> dict:
    refs_n = [normalise(r) for r in refs]
    hyps_n = [normalise(h) for h in hyps]
    w = wer(refs_n, hyps_n)
    c = cer(refs_n, hyps_n)
    return {"wer": round(w, 4), "cer": round(c, 4), "score": round(0.5 * w + 0.5 * c, 4)}


if TEXT_COL is not None:
    rng = np.random.default_rng(SEED)
    for lang in LANGS:
        lang_rows = train_df[
            (train_df[LANG_COL] == lang) & (train_df[ID_COL].astype(str).isin(audio_index.keys()))
        ]
        if lang_rows.empty:
            print(f"{lang}: no matched rows in Train.csv, skipping sanity check")
            continue
        sample = lang_rows.sample(min(30, len(lang_rows)), random_state=SEED)
        arrays = [get_array(audio_index[str(i)]) for i in sample[ID_COL]]
        refs = sample[TEXT_COL].tolist()
        hyps = transcribe(arrays, lang)
        result = score(refs, hyps)
        print(f"{lang}: n={len(sample)}  WER={result['wer']:.3f}  CER={result['cer']:.3f}  score={result['score']:.3f}")
        for r, h in list(zip(refs, hyps))[:3]:
            print(f"    ref: {r}")
            print(f"    hyp: {h}")
else:
    print("TEXT_COL not detected — skipping sanity check, fix ID/TEXT/LANG cols above.")

## 7. Run inference on Test.csv and build submission.csv

In [ ]:
predictions = {}
for lang in LANGS:
    lang_rows = test_df[
        (test_df[LANG_COL] == lang) & (test_df[ID_COL].astype(str).isin(audio_index.keys()))
    ]
    if lang_rows.empty:
        print(f"{lang}: no matched rows in Test.csv")
        continue
    ids = lang_rows[ID_COL].astype(str).tolist()
    arrays = [get_array(audio_index[i]) for i in ids]
    hyps = transcribe(arrays, lang)
    for i, h in zip(ids, hyps):
        predictions[i] = normalise(h) or "."
    print(f"{lang}: transcribed {len(ids)} test rows")

missing = [i for i in test_df[ID_COL].astype(str) if i not in predictions]
if missing:
    print(f"WARNING: {len(missing)} test ids had no matched audio — filling with '.' placeholder")
    for i in missing:
        predictions[i] = "."

In [ ]:
submission = sample_sub_df.copy()
submission[SUB_ID_COL] = submission[SUB_ID_COL].astype(str)
submission[SUB_TEXT_COL] = submission[SUB_ID_COL].map(predictions)

assert submission[SUB_TEXT_COL].isna().sum() == 0, "Some submission rows have no prediction — check id matching."

out_path = f"{DATA_DIR}/submission_mms_zeroshot.csv"
submission.to_csv(out_path, index=False)
print(f"Saved {out_path}  shape={submission.shape}")
display(submission.head())

## 8. Submit to Zindi (optional — costs one of your 5 daily submissions)

Review `submission.csv` above first. Flip `DO_SUBMIT = True` to actually
submit.

In [ ]:
DO_SUBMIT = False

if DO_SUBMIT:
    user.submit(
        filepaths=[out_path],
        comments=["Zero-shot MMS-1b-all baseline, lin/lug/sna"],
    )
    print("Submitted.")
else:
    print("DO_SUBMIT is False — nothing submitted. Set it to True once you're happy with the sanity-check scores above.")